# 03 — Chạy PBE (PySCF) để lấy mật độ electron
PBE/cc-pVDZ tự hợp cho mọi hệ trong `systems.csv` (phân tử vỏ kín dùng RKS; nguyên tử cô lập dùng UKS ở spin cơ bản). Lưu mật độ ρ và |∇ρ| trên lưới tích phân, trọng số lưới, ma trận mật độ và năng lượng.

- Có **resume**: hệ đã có file `.json` được bỏ qua.
- `∫ρ` phải bằng số electron (sai số lưới < 1e-3), nếu không hệ bị đánh dấu lỗi.

**Đầu ra:** `data/processed/pbe/<name>.npz` và `<name>.json`.

In [ ]:
# Cài thư viện (chỉ trên Colab; runtime Colab bị reset sau mỗi phiên). Chạy cục bộ: pip install -r requirements.txt
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf

In [ ]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ml_xc_colab")
else:                                    # chạy cục bộ (VS Code/Jupyter): notebook nằm trong notebooks/
    here = Path(os.environ.get("ML_XC_ROOT", Path.cwd())).resolve()
    ROOT = next((p for p in (here, here.parent) if (p / "utils").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Không thấy thư mục utils/. Mở notebook trong ml_xc_colab/notebooks hoặc đặt ML_XC_ROOT.")
sys.path.insert(0, str(ROOT))
ROOT = str(ROOT)
DATA, RAW, PROC, CKPT = (f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed", f"{ROOT}/data/checkpoints")
for d in (RAW, PROC, CKPT):
    os.makedirs(d, exist_ok=True)
try:                                     # giới hạn bộ nhớ PySCF = 70% RAM (MB); đặt trước khi import pyscf
    import psutil
    os.environ.setdefault("PYSCF_MAX_MEMORY", str(int(psutil.virtual_memory().total / 1e6 * 0.7)))
except ImportError:
    pass
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd
from utils import dft_utils as dft

BASIS = "cc-pvdz"
GRID_LEVEL = 3
CONV_TOL = 1e-9
PBE_DIR = f"{PROC}/pbe"
os.makedirs(PBE_DIR, exist_ok=True)
FORCE_RERUN = False

systems = pd.read_csv(f"{PROC}/systems.csv")
systems = systems.sort_values(["is_atom", "n_atoms"], ascending=[False, True])     # nguyên tử trước, rồi phân tử nhỏ trước
print(dft.resources())
print(f"{len(systems)} hệ ({int(systems.is_atom.sum())} nguyên tử + {int((~systems.is_atom).sum())} phân tử), basis {BASIS}")

In [ ]:
t_start = time.time()
n_new = n_skip = n_fail = 0
for r in systems.itertuples():
    js, npz = f"{PBE_DIR}/{r.name}.json", f"{PBE_DIR}/{r.name}.npz"
    if os.path.exists(js) and os.path.exists(npz) and not FORCE_RERUN:
        n_skip += 1
        continue
    try:
        mol = dft.build_mol(json.loads(r.symbols_json), json.loads(r.coords_json), basis=BASIS, spin=int(r.spin))
        info, mf = dft.run_ks(mol, "pbe", grid_level=GRID_LEVEL, conv_tol=CONV_TOL)
        den = dft.density_on_grid(mol, mf)
        info.update(name=r.name, basis=BASIS, int_rho=den["int_rho"], n_grid=int(len(den["rho"])),
                    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
        np.savez_compressed(npz, coords=den["coords"].astype(np.float32), weights=den["weights"],
                            rho=den["rho"], gnorm=den["gnorm"], dm=den["dm"])
        json.dump(info, open(js, "w"), indent=2)          # ghi json sau npz: có json nghĩa là đã xong
        n_new += 1
        flag = "" if info["converged"] and abs(info["int_rho"] - info["nelec"]) < 1e-3 else "  <-- KIỂM TRA"
        print(f"{r.name:14s} {r.formula:10s} E={info['e_tot']:.6f} Ha  conv={info['converged']}  "
              f"∫ρ={info['int_rho']:.5f}/{info['nelec']}  grid={info['n_grid']}  {info['wall_s']}s{flag}")
    except Exception as e:
        n_fail += 1
        print(f"{r.name:14s} {r.formula:10s} LỖI: {type(e).__name__}: {str(e)[:120]}")
print(f"\nXong: mới {n_new}, bỏ qua {n_skip}, lỗi {n_fail} | {(time.time()-t_start)/60:.1f} phút")

## Tổng hợp

In [ ]:
recs = [json.load(open(f"{PBE_DIR}/{r.name}.json")) for r in systems.itertuples() if os.path.exists(f"{PBE_DIR}/{r.name}.json")]
st = pd.DataFrame(recs)
print(f"Có kết quả: {len(st)}/{len(systems)} | hội tụ: {int(st.converged.sum())} | tổng thời gian: {st.wall_s.sum()/60:.1f} phút")
bad = st[(~st.converged) | ((st.int_rho - st.nelec).abs() > 1e-3)]
print(bad[["name", "converged", "int_rho", "nelec"]].to_string(index=False) if len(bad) else "Mọi hệ đạt (hội tụ và ∫ρ đúng)")
st[["name", "e_tot", "exc", "nbf", "n_grid", "wall_s"]].describe().round(3)